# chrome

> Shared chrome switching route handlers for the combined Phase 2 step

In [ ]:
#| default_exp routes.chrome

In [ ]:
#| export
from typing import Tuple, Dict, Callable

from fasthtml.common import APIRouter, Div

from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcript_segment_align.html_ids import CombinedHtmlIds
from cjm_transcript_segmentation.models import TextSegment, SegmentationUrls
from cjm_transcript_vad_align.models import VADChunk, AlignmentUrls

# Segmentation renderers
from cjm_transcript_segmentation.components.step_renderer import (
    render_toolbar as render_seg_toolbar,
    render_seg_footer_content,
)
from cjm_transcript_segmentation.components.card_stack_config import (
    SEG_CS_CONFIG, SEG_CS_IDS,
)

# Alignment renderers
from cjm_transcript_vad_align.components.step_renderer import (
    render_align_toolbar, render_align_footer_content,
)
from cjm_transcript_vad_align.components.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS,
)

# Footer helper + FA extra_actions builder + match detection
from cjm_transcript_segment_align.components.step_renderer import (
    render_footer_inner_content,
)
from cjm_transcript_segment_align.components.handlers import (
    build_fa_extra_actions, segments_match_presplit,
)

# Keyboard config
from cjm_transcript_segment_align.components.keyboard_config import (
    build_combined_kb_system, render_keyboard_hints_collapsible,
)

DEBUG_SWITCH_CHROME = False

## Handlers

In [ ]:
#| export
async def _handle_switch_chrome(
    state_store:SQLiteWorkflowStateStore,  # State store instance
    workflow_id:str,  # Workflow identifier
    request,  # FastHTML request object
    sess,  # FastHTML session object
    seg_urls:SegmentationUrls,  # URL bundle for segmentation routes
    align_urls:AlignmentUrls,  # URL bundle for alignment routes
    jm_trigger=None,  # Pre-rendered job monitor trigger element
    fa_toggle_url:str="",  # URL for FA toggle route
    fa_available:bool=False,  # Whether FA plugin is available
) -> tuple:  # OOB swaps for shared chrome containers
    """Switch shared chrome content based on active column."""
    form = await request.form()
    active_column = form.get("active_column", "seg")

    if DEBUG_SWITCH_CHROME:
        print(f"[SWITCH_CHROME] active_column: {active_column}")

    session_id = get_session_id(sess)
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    seg_state = step_states.get("segmentation", {})
    align_state = step_states.get("alignment", {})

    # Get counts for alignment status
    segment_count = len(seg_state.get("segments", []))
    chunk_count = len(align_state.get("vad_chunks", []))

    # Build combined KB manager for hints
    kb_manager, _ = build_combined_kb_system(seg_urls, align_urls)

    if active_column == "seg":
        # Segmentation chrome
        segments = [TextSegment.from_dict(s) for s in seg_state.get("segments", [])]
        history = seg_state.get("history", [])
        focused_index = seg_state.get("focused_index", 0)
        visible_count = seg_state.get("visible_count", DEFAULT_VISIBLE_COUNT)
        is_auto_mode = seg_state.get("is_auto_mode", False)
        card_width = seg_state.get("card_width", DEFAULT_CARD_WIDTH)

        # Build FA extra_actions and NLTK Split disabled state
        fa_extra = build_fa_extra_actions(
            seg_state, jm_trigger, fa_toggle_url, fa_available,
        )
        nltk_presplit = seg_state.get("nltk_presplit", [])
        nltk_disabled = segments_match_presplit(
            seg_state.get("segments", []), nltk_presplit,
        )

        hints_content = render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True)
        toolbar_content = render_seg_toolbar(
            reset_url=seg_urls.reset,
            ai_split_url=seg_urls.ai_split,
            undo_url=seg_urls.undo,
            can_undo=(len(history) > 0),
            visible_count=visible_count,
            is_auto_mode=is_auto_mode,
            extra_actions=fa_extra,
            nltk_split_disabled=nltk_disabled,
        )
        controls_content = render_width_slider(SEG_CS_CONFIG, SEG_CS_IDS, card_width=card_width)
        column_footer = render_seg_footer_content(segments, focused_index)
    else:
        # Alignment chrome
        chunks = [VADChunk.from_dict(c) for c in align_state.get("vad_chunks", [])]
        focused_index = align_state.get("focused_chunk_index", 0)
        visible_count = align_state.get("visible_count", 5)
        is_auto_mode = align_state.get("is_auto_mode", False)
        card_width = align_state.get("card_width", 40)

        hints_content = render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True)
        toolbar_content = render_align_toolbar(
            visible_count=visible_count,
            is_auto_mode=is_auto_mode,
        )
        controls_content = render_width_slider(ALIGN_CS_CONFIG, ALIGN_CS_IDS, card_width=card_width)
        column_footer = render_align_footer_content(chunks, focused_index)

    if DEBUG_SWITCH_CHROME:
        print(f"[SWITCH_CHROME] returning OOB swaps for {active_column}")

    # Return OOB swaps
    hints_oob = Div(
        hints_content,
        id=CombinedHtmlIds.SHARED_HINTS,
        hx_swap_oob="innerHTML"
    )
    toolbar_oob = Div(
        toolbar_content,
        id=CombinedHtmlIds.SHARED_TOOLBAR,
        hx_swap_oob="innerHTML"
    )
    controls_oob = Div(
        controls_content,
        id=CombinedHtmlIds.SHARED_CONTROLS,
        hx_swap_oob="innerHTML"
    )
    footer_oob = Div(
        render_footer_inner_content(column_footer, segment_count, chunk_count),
        id=CombinedHtmlIds.SHARED_FOOTER,
        hx_swap_oob="innerHTML"
    )

    return (hints_oob, toolbar_oob, controls_oob, footer_oob)

## Router

In [ ]:
#| export
def init_chrome_router(
    state_store: SQLiteWorkflowStateStore,  # State store instance
    workflow_id: str,  # Workflow identifier
    seg_urls: SegmentationUrls,  # URL bundle for segmentation routes
    align_urls: AlignmentUrls,  # URL bundle for alignment routes
    prefix: str,  # Route prefix (e.g., "/workflow/core/chrome")
    jm_trigger = None,  # Pre-rendered job monitor trigger element
    fa_toggle_url: str = "",  # URL for FA toggle route
    fa_available: bool = False,  # Whether FA plugin is available
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize chrome switching routes."""
    router = APIRouter(prefix=prefix)

    @router
    async def switch_chrome(request, sess):
        """Switch shared chrome content based on active column."""
        return await _handle_switch_chrome(
            state_store, workflow_id, request, sess,
            seg_urls=seg_urls,
            align_urls=align_urls,
            jm_trigger=jm_trigger,
            fa_toggle_url=fa_toggle_url,
            fa_available=fa_available,
        )

    routes = {
        "switch_chrome": switch_chrome,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()